# AI/ML Recruitment Task - First Year
## Auto MPG: EDA and Linear Regression

Name: ____________________

This notebook explores the Auto MPG dataset, cleans it, and uses linear regression to predict MPG. Run all cells from top to bottom.

## Task 1: Load and inspect the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"
df = pd.read_csv(url)
df = df.rename(columns={"name": "car_name"})

print(df.head())
print("Rows and columns:", df.shape)
print(df.info())
print(df.describe())

## Clean the data
I remove exact duplicates and fill missing horsepower values with the median.

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

numeric_cols = ["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["origin"] = pd.to_numeric(df["origin"], errors="coerce")
df = df.dropna(subset=["mpg", "weight"])
print("Shape after cleaning:", df.shape)
print(df.isnull().sum())
df.to_csv("auto_mpg_cleaned.csv", index=False)
print("Saved auto_mpg_cleaned.csv")

## Explore MPG and vehicle details

In [ ]:
print(df["mpg"].describe())
sns.histplot(df["mpg"], bins=20, kde=True)
plt.title("Distribution of MPG")
plt.show()

cols = ["displacement", "horsepower", "weight", "acceleration"]
df[cols].hist(figsize=(9, 6), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
for col in ["weight", "horsepower", "displacement", "acceleration"]:
    plt.figure(figsize=(5, 3.5))
    sns.scatterplot(data=df, x=col, y="mpg")
    plt.title("MPG compared with " + col)
    plt.show()

sns.boxplot(data=df, x="cylinders", y="mpg")
plt.title("MPG by cylinders")
plt.show()
sns.boxplot(data=df, x="model_year", y="mpg")
plt.title("MPG by model year")
plt.xticks(rotation=45)
plt.show()

## Correlation and unusual values

In [ ]:
corr_cols = numeric_cols + ["origin"]
corr = df[corr_cols].corr()
print(corr["mpg"].sort_values(ascending=False))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation between numeric columns")
plt.show()

sns.boxplot(y=df["mpg"])
plt.title("Possible unusual MPG values")
plt.show()
print(df.sort_values("mpg").head(3)[["mpg", "weight", "horsepower", "car_name"]])
print(df.sort_values("mpg").tail(3)[["mpg", "weight", "horsepower", "car_name"]])

## My own question
Do newer model years tend to have higher MPG? I compare average and median MPG for earlier and later years.

In [ ]:
middle_year = df["model_year"].median()
df["year_group"] = np.where(df["model_year"] <= middle_year, "Earlier years", "Later years")
print(df.groupby("year_group")["mpg"].agg(["mean", "median", "count"]))
sns.boxplot(data=df, x="year_group", y="mpg")
plt.title("MPG in earlier and later model years")
plt.show()

## Task 1 findings
Use the printed values and graphs to write 5-8 findings in your own words. Include the final dataset shape, missing values and duplicates, MPG distribution, relationships with other variables, group comparisons, one unusual record, and your conclusion for the model-year question.

## Task 2: Linear Regression
First I use vehicle weight only, then several features. Origin is one-hot encoded, and car_name is not used.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y = df["mpg"]
X_weight = df[["weight"]]
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_weight, y, test_size=0.2, random_state=42)
weight_model = LinearRegression()
weight_model.fit(Xw_train, yw_train)
weight_pred = weight_model.predict(Xw_test)
print("Weight-only test R2:", r2_score(yw_test, weight_pred))

In [ ]:
features = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year", "origin"]
X = df[features].copy()
X["origin"] = X["origin"].astype("category")
X = pd.get_dummies(X, columns=["origin"], drop_first=True, dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

## Evaluate the model
MAE is the average absolute error in MPG. MSE gives larger errors more weight. RMSE is in MPG units. R-squared describes how much variation is explained by the model.

In [ ]:
def show_scores(actual, predicted, label):
    mse = mean_squared_error(actual, predicted)
    print(label)
    print("MAE:", round(mean_absolute_error(actual, predicted), 2))
    print("MSE:", round(mse, 2))
    print("RMSE:", round(np.sqrt(mse), 2))
    print("R2:", round(r2_score(actual, predicted), 2))

show_scores(y_train, train_pred, "Training scores")
show_scores(y_test, test_pred, "Testing scores")

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, test_pred)
plt.xlabel("Actual MPG")
plt.ylabel("Predicted MPG")
plt.title("Actual and predicted MPG")
low = min(y_test.min(), test_pred.min())
high = max(y_test.max(), test_pred.max())
plt.plot([low, high], [low, high], color="red")
plt.show()

## Task 2 findings and limitations
Use the scores and plot to write 4-6 findings. Compare training and test scores to comment on overfitting or underfitting. Limitations: linear regression may miss curved patterns, vehicle features can be related, the dataset is small, and some useful details are missing. A next step could be comparing with a decision tree.